# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [15]:
# YOUR CHANGES HERE
import pandas as pd
import numpy as np

# ---- Load data ----
TRANSITIONS_PATH = "twizzleflu-transitions.tsv"
REWARDS_PATH     = "twizzleflu-rewards.tsv"

trans = pd.read_csv(TRANSITIONS_PATH, sep="\t")
rews  = pd.read_csv(REWARDS_PATH, sep="\t")

# ---- Evaluate fixed policy: always "do-nothing" ----
POLICY_ACTION = "do-nothing"

# States (union from transitions + rewards)
states = sorted(set(trans["state"]).union(trans["next_state"]).union(rews["state"]))

# Reward function r(s, a) -> reward
reward_map = {(row.state, row.action): float(row.reward) for row in rews.itertuples(index=False)}
def r(s, a):
    # default 0 if not explicitly listed
    return reward_map.get((s, a), 0.0)

# Transition probabilities P(s'|s,a)
P = {s: [] for s in states}
sub = trans[trans["action"] == POLICY_ACTION]
for row in sub.itertuples(index=False):
    P[row.state].append((row.next_state, float(row.probability)))

# Safety: if a state has no outgoing transitions under the policy, make it absorbing
for s in states:
    if len(P[s]) == 0:
        P[s] = [(s, 1.0)]

# ---- Iterative Bellman policy evaluation (undiscounted) ----
V = {s: 0.0 for s in states}  # initial guess
tol = 1e-12
max_iter = 1_000_000

for it in range(max_iter):
    delta = 0.0
    V_new = {}
    for s in states:
        v = r(s, POLICY_ACTION) + sum(p * V[s2] for (s2, p) in P[s])
        V_new[s] = v
        delta = max(delta, abs(v - V[s]))
    V = V_new
    if delta < tol:
        # print(f"Converged in {it+1} iterations, delta={delta:.3e}")
        break
else:
    raise RuntimeError("Did not converge—check if the policy reaches an absorbing state with prob 1.")

# ---- Output: values + expected discomfort from a start state ----
start_state = "exposed-1"
expected_reward = V[start_state]
expected_discomfort = -expected_reward  # discomfort is negative reward

print("State values V^pi(s) under always do-nothing:")
for s in states:
    print(f"  {s:>11s}: {V[s]: .10f}")

print(f"\nStart state: {start_state}")
print(f"Expected reward:     {expected_reward:.10f}")
print(f"Expected discomfort: {expected_discomfort:.10f}")

...

State values V^pi(s) under always do-nothing:
    exposed-1: -3.4133333333
    exposed-2: -4.2666666667
    exposed-3: -5.3333333333
    recovered:  0.0000000000
   symptoms-1: -6.6666666667
   symptoms-2: -5.0000000000
   symptoms-3: -1.6666666667

Start state: exposed-1
Expected reward:     -3.4133333333
Expected discomfort: 3.4133333333


Ellipsis

Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [16]:
# YOUR CHANGES HERE

# ---- Save expected discomfort by state ----
output_path = "do-nothing-discomfort.tsv"

df_out = pd.DataFrame({
    "state": states,
    "expected_discomfort": [-V[s] for s in states]  # discomfort = - reward
})

df_out.to_csv(output_path, sep="\t", index=False)

print(f"Saved expected discomforts to {output_path}")


Saved expected discomforts to do-nothing-discomfort.tsv


Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [17]:
# YOUR CHANGES HERE
states = sorted(set(trans["state"]).union(trans["next_state"]).union(rews["state"]))
actions = sorted(trans["action"].unique())

# Reward function r(s,a)
reward_map = {(row.state, row.action): float(row.reward) for row in rews.itertuples(index=False)}
def r(s, a):
    return reward_map.get((s, a), 0.0)

# Transition model P[(s,a)] = [(s',p), ...]
P = {}
for row in trans.itertuples(index=False):
    P.setdefault((row.state, row.action), []).append((row.next_state, float(row.probability)))

# ---- Value Iteration for optimal policy (undiscounted, gamma=1) ----
tol = 1e-12
max_iter = 1_000_000

V = {s: 0.0 for s in states}
pi = {s: actions[0] for s in states}

for it in range(max_iter):
    delta = 0.0
    V_new = {}
    pi_new = {}

    for s in states:
        best_val = -1e18
        best_a = actions[0]

        # Bellman optimality update: V(s) = max_a [ r(s,a) + sum_s' P(s'|s,a) V(s') ]
        for a in actions:
            q = r(s, a) + sum(p * V[s2] for (s2, p) in P[(s, a)])
            # deterministic tie-break by action order
            if q > best_val + 1e-15:
                best_val = q
                best_a = a

        V_new[s] = best_val
        pi_new[s] = best_a
        delta = max(delta, abs(best_val - V[s]))

    V, pi = V_new, pi_new
    if delta < tol:
        # print(f"Converged in {it+1} iterations, delta={delta:.3e}")
        break
else:
    raise RuntimeError("Did not converge. Check that the MDP reaches 'recovered' with probability 1.")

# ---- Results ----
start_state = "exposed-1"
expected_reward = V[start_state]
expected_discomfort = -expected_reward  # discomfort = -reward

print("Optimal policy (min discomfort / max reward):")
for s in states:
    print(f"  {s:>11s}: {pi[s]}")

print(f"\nStart state: {start_state}")
print(f"Expected reward under optimal policy:     {expected_reward:.10f}")
print(f"Expected discomfort under optimal policy: {expected_discomfort:.10f}")

# ---- (Optional but useful) Save policy + discomforts ----
out_path = "optimal-treatment-plan.tsv"
df_out = pd.DataFrame({
    "state": states,
    "optimal_action": [pi[s] for s in states],
    "expected_discomfort": [-V[s] for s in states],
})
df_out.to_csv(out_path, sep="\t", index=False)
print(f"\nSaved optimal plan to {out_path}")


Optimal policy (min discomfort / max reward):
    exposed-1: sleep-8
    exposed-2: sleep-8
    exposed-3: sleep-8
    recovered: do-nothing
   symptoms-1: drink-oj
   symptoms-2: drink-oj
   symptoms-3: drink-oj

Start state: exposed-1
Expected reward under optimal policy:     -0.7500000000
Expected discomfort under optimal policy: 0.7500000000

Saved optimal plan to optimal-treatment-plan.tsv


Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [18]:
# YOUR CHANGES HERE

# ---- Save optimal actions ----
output_path = "minimum-discomfort-actions.tsv"

df_actions = pd.DataFrame({
    "state": states,
    "action": [pi[s] for s in states]
})

df_actions.to_csv(output_path, sep="\t", index=False)

print(f"Saved optimal actions to {output_path}")


Saved optimal actions to minimum-discomfort-actions.tsv


Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [19]:
# YOUR CHANGES HERE


states = sorted(set(trans["state"]).union(trans["next_state"]).union(rews["state"]))
actions = sorted(trans["action"].unique())

# Reward function r(s,a)
reward_map = {(row.state, row.action): float(row.reward) for row in rews.itertuples(index=False)}
def r(s, a):
    return reward_map.get((s, a), 0.0)

# Transition model P[(s,a)] = [(s',p), ...]
P = {}
for row in trans.itertuples(index=False):
    P.setdefault((row.state, row.action), []).append((row.next_state, float(row.probability)))

# ---------------------------
# STEP 1: Compute optimal policy pi via value iteration (Part 2)
# ---------------------------
tol = 1e-12
max_iter = 1_000_000

V = {s: 0.0 for s in states}
pi = {s: actions[0] for s in states}

for it in range(max_iter):
    delta = 0.0
    V_new = {}
    pi_new = {}
    for s in states:
        best_val = -1e18
        best_a = actions[0]
        for a in actions:
            q = r(s, a) + sum(p * V[s2] for (s2, p) in P[(s, a)])
            if q > best_val + 1e-15:
                best_val = q
                best_a = a
        V_new[s] = best_val
        pi_new[s] = best_a
        delta = max(delta, abs(best_val - V[s]))
    V, pi = V_new, pi_new
    if delta < tol:
        break
else:
    raise RuntimeError("Value iteration did not converge.")

# ---------------------------
# STEP 2: Evaluate the optimal policy (Part 3)
# ---------------------------
V_pi = {s: 0.0 for s in states}  # re-init values for policy evaluation

for it in range(max_iter):
    delta = 0.0
    V_new = {}
    for s in states:
        a = pi[s]
        v = r(s, a) + sum(p * V_pi[s2] for (s2, p) in P[(s, a)])
        V_new[s] = v
        delta = max(delta, abs(v - V_pi[s]))
    V_pi = V_new
    if delta < tol:
        break
else:
    raise RuntimeError("Policy evaluation did not converge.")

# Expected discomfort per state = - V^pi(s)
discomfort = {s: -V_pi[s] for s in states}

print("Expected discomfort by state under the optimal policy:")
for s in states:
    print(f"  {s:>11s}: {discomfort[s]: .10f}")

# ---- (Optional) Save to TSV for submission ----
output_path = "minimum-discomfort-discomfort.tsv"
df_out = pd.DataFrame({
    "state": states,
    "expected_discomfort": [discomfort[s] for s in states]
})
df_out.to_csv(output_path, sep="\t", index=False)
print(f"\nSaved to {output_path}")


Expected discomfort by state under the optimal policy:
    exposed-1:  0.7500000000
    exposed-2:  1.5000000000
    exposed-3:  3.0000000000
    recovered: -0.0000000000
   symptoms-1:  6.0000000000
   symptoms-2:  4.5000000000
   symptoms-3:  1.5000000000

Saved to minimum-discomfort-discomfort.tsv


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [20]:
# YOUR CHANGES HERE

output_path = "minimum-discomfort-values.tsv"

df_out = pd.DataFrame({
    "state": states,
    "expected_discomfort": [discomfort[s] for s in states]
})

df_out.to_csv(output_path, sep="\t", index=False)

print(f"Saved expected discomfort values to {output_path}")


Saved expected discomfort values to minimum-discomfort-values.tsv


Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [29]:
# YOUR CHANGES HERE

states = sorted(set(trans["state"]).union(trans["next_state"]))
actions = sorted(trans["action"].unique())

# ---- Define NEW reward: -1 per day in "sick" (symptoms-*), 0 otherwise ----
# exposed-* does NOT count as sick; recovered is 0; action does not matter.
def r_duration(s, a):
    return -1.0 if str(s).startswith("symptoms-") else 0.0

# Transition model P[(s,a)] = [(s',p), ...]
P = {}
for row in trans.itertuples(index=False):
    P.setdefault((row.state, row.action), []).append((row.next_state, float(row.probability)))

# ---- Value Iteration (gamma=1) to minimize duration / maximize rewards ----
tol = 1e-12
max_iter = 1_000_000

V = {s: 0.0 for s in states}
pi = {s: actions[0] for s in states}

for it in range(max_iter):
    delta = 0.0
    V_new = {}
    pi_new = {}

    for s in states:
        best_val = -1e18
        best_a = actions[0]

        for a in actions:
            q = r_duration(s, a) + sum(p * V[s2] for (s2, p) in P[(s, a)])
            if q > best_val + 1e-15:
                best_val = q
                best_a = a

        V_new[s] = best_val
        pi_new[s] = best_a
        delta = max(delta, abs(best_val - V[s]))

    V, pi = V_new, pi_new
    if delta < tol:
        break
else:
    raise RuntimeError("Did not converge. Check that 'recovered' is absorbing and reachable w.p. 1.")

# ---- Interpret values ----
# With reward -1 per day in symptoms, -V(s) = expected number of sick days remaining from state s.
expected_sick_days = {s: -V[s] for s in states}

print("Policy that minimizes Twizzleflu symptom duration (expected sick days):")
for s in states:
    print(f"  {s:>11s}: action={pi[s]:<12s}  expected_sick_days={expected_sick_days[s]:.10f}")

# ---- Optional: save plan + expected sick days ----
df_out = pd.DataFrame({
    "state": states,
    "action": [pi[s] for s in states],
    "expected_sick_days": [expected_sick_days[s] for s in states],
})
df_out.to_csv("minimum-duration-plan.tsv", sep="\t", index=False)
print("\nSaved to minimum-duration-plan.tsv")


Policy that minimizes Twizzleflu symptom duration (expected sick days):
    exposed-1: action=sleep-8       expected_sick_days=1.2500000000
    exposed-2: action=sleep-8       expected_sick_days=2.5000000000
    exposed-3: action=sleep-8       expected_sick_days=5.0000000000
    recovered: action=do-nothing    expected_sick_days=-0.0000000000
   symptoms-1: action=do-nothing    expected_sick_days=10.0000000000
   symptoms-2: action=do-nothing    expected_sick_days=6.6666666667
   symptoms-3: action=do-nothing    expected_sick_days=3.3333333333

Saved to minimum-duration-plan.tsv


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [30]:
# YOUR CHANGES HERE


# Unique states and actions (matching original reward file structure)
states = sorted(set(trans["state"]).union(trans["next_state"]))
actions = sorted(trans["action"].unique())

# Build new reward table
rows = []
for s in states:
    for a in actions:
        reward = -1.0 if str(s).startswith("symptoms-") else 0.0
        rows.append({
            "state": s,
            "action": a,
            "reward": reward
        })

duration_rewards = pd.DataFrame(rows)

# Save in same format as twizzleflu-rewards.tsv
output_path = "duration-rewards.tsv"
duration_rewards.to_csv(output_path, sep="\t", index=False)
print(f"Saved duration reward function to {output_path}")


Saved duration reward function to duration-rewards.tsv


Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [31]:
# YOUR CHANGES HERE

states = sorted(set(trans["state"]).union(trans["next_state"]))
actions = sorted(trans["action"].unique())

# Transition model P[(s,a)] = [(s',p), ...]
P = {}
for row in trans.itertuples(index=False):
    P.setdefault((row.state, row.action), []).append((row.next_state, float(row.probability)))

# New duration reward
def r_duration(s, a):
    return -1.0 if str(s).startswith("symptoms-") else 0.0

# Value iteration (gamma=1)
tol = 1e-12
max_iter = 1_000_000

V = {s: 0.0 for s in states}
pi = {s: actions[0] for s in states}

for it in range(max_iter):
    delta = 0.0
    V_new = {}
    pi_new = {}

    for s in states:
        best_val = -1e18
        best_a = actions[0]

        for a in actions:
            # Bellman optimality update
            q = r_duration(s, a) + sum(p * V[s2] for (s2, p) in P[(s, a)])
            if q > best_val + 1e-15:  # deterministic tie-break
                best_val = q
                best_a = a

        V_new[s] = best_val
        pi_new[s] = best_a
        delta = max(delta, abs(best_val - V[s]))

    V, pi = V_new, pi_new
    if delta < tol:
        # print(f"Converged in {it+1} iterations, delta={delta:.3e}")
        break
else:
    raise RuntimeError("Did not converge. Check that recovery is absorbing and reachable w.p. 1.")

# Interpretation: expected symptom-days remaining from state s is -V(s)
expected_symptom_days = {s: -V[s] for s in states}

print("Optimal policy to minimize duration (symptom days):")
for s in states:
    print(f"  {s:>11s}: action={pi[s]:<12s}  expected_symptom_days={expected_symptom_days[s]:.10f}")

# Optional: save actions (uncomment if you want a TSV)
# df_actions = pd.DataFrame({"state": states, "action": [pi[s] for s in states]})
# df_actions.to_csv("minimum-duration-actions.tsv", sep="\t", index=False)


Optimal policy to minimize duration (symptom days):
    exposed-1: action=sleep-8       expected_symptom_days=1.2500000000
    exposed-2: action=sleep-8       expected_symptom_days=2.5000000000
    exposed-3: action=sleep-8       expected_symptom_days=5.0000000000
    recovered: action=do-nothing    expected_symptom_days=-0.0000000000
   symptoms-1: action=do-nothing    expected_symptom_days=10.0000000000
   symptoms-2: action=do-nothing    expected_symptom_days=6.6666666667
   symptoms-3: action=do-nothing    expected_symptom_days=3.3333333333


Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [32]:
# YOUR CHANGES HERE

# ---- Save optimal duration-minimizing actions ----
output_path = "minimum-duration-actions.tsv"

df_actions = pd.DataFrame({
    "state": states,
    "action": [pi[s] for s in states]
})

df_actions.to_csv(output_path, sep="\t", index=False)

print(f"Saved optimal duration actions to {output_path}")


Saved optimal duration actions to minimum-duration-actions.tsv


Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [34]:
# YOUR CHANGES HERE

expected_symptom_days = {s: -V[s] for s in states}


Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [35]:
# YOUR CHANGES HERE

# ---- Save expected sick days per state ----
output_path = "minimum-duration-values.tsv"

df_out = pd.DataFrame({
    "state": states,
    "expected_days_sick": [expected_symptom_days[s] for s in states]
})

df_out.to_csv(output_path, sep="\t", index=False)

print(f"Saved expected sick days to {output_path}")


Saved expected sick days to minimum-duration-values.tsv


Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

In [36]:
# YOUR CHANGES HERE
states  = sorted(set(trans["state"]).union(trans["next_state"]).union(rews["state"]))
actions = sorted(trans["action"].unique())

# Original reward function r(s,a)
reward_map = {(row.state, row.action): float(row.reward) for row in rews.itertuples(index=False)}
def r_original(s, a):
    return reward_map.get((s, a), 0.0)

# Transitions P[(s,a)] = [(s',p), ...]
P = {}
for row in trans.itertuples(index=False):
    P.setdefault((row.state, row.action), []).append((row.next_state, float(row.probability)))

# Duration reward: -1 if state is symptoms-*, else 0 (action doesn't matter)
def r_duration(s, a):
    return -1.0 if str(s).startswith("symptoms-") else 0.0

# ----------------------------
# Helpers: value iteration + policy evaluation (gamma=1)
# ----------------------------
def value_iteration_opt(reward_fn, tol=1e-12, max_iter=1_000_000):
    V = {s: 0.0 for s in states}
    pi = {s: actions[0] for s in states}

    for _ in range(max_iter):
        delta = 0.0


Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [40]:
import pandas as pd

states  = sorted(set(trans["state"]).union(trans["next_state"]).union(rews["state"]))
actions = sorted(trans["action"].unique())

# ----------------------------
# Build transition model
# ----------------------------
P = {}
for row in trans.itertuples(index=False):
    P.setdefault((row.state, row.action), []).append((row.next_state, float(row.probability)))

# ----------------------------
# Reward functions
# ----------------------------
reward_map = {(row.state, row.action): float(row.reward) for row in rews.itertuples(index=False)}

def r_original(s, a):
    return reward_map.get((s, a), 0.0)

def r_duration(s, a):
    return -1.0 if str(s).startswith("symptoms-") else 0.0

# ----------------------------
# Value Iteration
# ----------------------------
def value_iteration(reward_fn, tol=1e-12, max_iter=1_000_000):
    V = {s: 0.0 for s in states}
    pi = {s: actions[0] for s in states}

    for _ in range(max_iter):
        delta = 0.0
        V_new = {}
        pi_new = {}

        for s in states:
            best_val = -1e18
            best_a = actions[0]

            for a in actions:
                q = reward_fn(s, a) + sum(p * V[s2] for (s2, p) in P[(s, a)])
                if q > best_val + 1e-15:
                    best_val = q
                    best_a = a

            V_new[s] = best_val
            pi_new[s] = best_a
            delta = max(delta, abs(best_val - V[s]))

        V, pi = V_new, pi_new
        if delta < tol:
            return V, pi

    raise RuntimeError("Value iteration did not converge.")

# ----------------------------
# Policy Evaluation
# ----------------------------
def policy_evaluation(pi, reward_fn, tol=1e-12, max_iter=1_000_000):
    V = {s: 0.0 for s in states}

    for _ in range(max_iter):
        delta = 0.0
        V_new = {}
        for s in states:
            a = pi[s]
            v = reward_fn(s, a) + sum(p * V[s2] for (s2, p) in P[(s, a)])
            V_new[s] = v
            delta = max(delta, abs(v - V[s]))
        V = V_new
        if delta < tol:
            return V

    raise RuntimeError("Policy evaluation did not converge.")

# ----------------------------
# Compute both policies
# ----------------------------
_, pi_min_discomfort = value_iteration(r_original)
_, pi_min_duration   = value_iteration(r_duration)

# ----------------------------
# Compute expected discomfort under both policies
# (using ORIGINAL rewards)
# ----------------------------
V_discomfort_policy = policy_evaluation(pi_min_discomfort, r_original)
V_duration_policy   = policy_evaluation(pi_min_duration, r_original)

expected_discomfort_min_discomfort = {s: -V_discomfort_policy[s] for s in states}
expected_discomfort_min_duration   = {s: -V_duration_policy[s]   for s in states}

# ----------------------------
# Save comparison file
# ----------------------------
df_compare = pd.DataFrame({
    "state": states,
    "speed_discomfort": [expected_discomfort_min_duration[s] for s in states],
    "minimize_discomfort": [expected_discomfort_min_discomfort[s] for s in states]
})

output_path = "policy-comparison.tsv"
df_compare.to_csv(output_path, sep="\t", index=False)

print(f"Saved policy comparison to {output_path}")


Saved policy comparison to policy-comparison.tsv


Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.